# Retrieval-Augmented Generation (RAG), End to End

Ingest -> parse -> chunk -> embed -> index -> retrieve (dense, BM25, hybrid) -> rerank -> build context ->
generate with citations -> evaluate. The embedder is a deterministic *lexical hashing* stand-in so the
notebook is offline; swap in a neural embedding model for semantic matching. The pipeline logic is unchanged.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('06-generative-ai/rag/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import rag_pipeline
print('rag_pipeline loaded')

## 1. Ingestion: parse, chunk, attach metadata

In [ ]:
from rag_pipeline import *
docs = {
 'billing': '''# Billing
## Refunds
Customers may request a refund within 30 days of purchase. Refunds are issued to the original payment method.
## Invoices
Invoices are emailed monthly. Error code ERR-4012 means the invoice address is missing.
''',
 'security': '''# Security
## Passwords
Passwords must be rotated every 90 days and must contain at least twelve characters.
## Incidents
Report suspected phishing to the security desk within one hour. Do not forward the suspicious email.
''',
 'travel': '''# Travel
## Booking
Flights must be booked through the corporate portal at least 14 days in advance.
## Expenses
Meal expenses are reimbursed up to 50 dollars per day with itemised receipts.
''',
}
chunks = []
for name, text in docs.items():
    chunks += ingest_markdown(name, text, size=25, overlap=5, extra_meta={'year': 2026})
print(len(chunks), 'chunks; example metadata:', chunks[0].metadata)

## 2. Index and search: dense vs. BM25 vs. hybrid (RRF)
Exact identifiers (e.g. `ERR-4012`) favour lexical search; fusing rankings is robust to either signal being weak.

In [ ]:
store = VectorStore(HashingEmbedder(512)); store.add(chunks)
bm = BM25(chunks)
q = 'what does ERR-4012 mean'
print('dense :', [c.id for c, _ in store.search(q, 3)])
print('bm25  :', [c.id for c, _ in bm.search(q, 3)])
print('hybrid:', [c.id for c in hybrid_search(store, bm, q, 3)])
print('filtered to security:', [c.id for c, _ in store.search('rotate passwords', 3, where={'source': 'security'})])

## 3. Rerank and diversify
A second stage rescoring top candidates (cross-encoder in production) and MMR to avoid near-duplicates.

In [ ]:
import numpy as np
cands = hybrid_search(store, bm, 'how often must passwords be rotated', 6)
print('before rerank:', [c.id for c in cands])
print('after rerank :', [c.id for c in rerank('how often must passwords be rotated', cands)])
emb = HashingEmbedder(512)
vecs = np.vstack([emb.embed(c.text) for c in cands])
sel = mmr(emb.embed('expense and travel rules'), vecs, 3, lam=0.6)
print('MMR picks:', [cands[i].id for i in sel])

## 4. Context construction, generation and citation checks
A mock generator stands in for an LLM: it quotes the best source and cites it. The **application** validates citations.

In [ ]:
question = 'Within how many days can customers request a refund?'
top = rerank(question, hybrid_search(store, bm, question, 5))
context, used = build_context(top, max_words=80)
print(rag_prompt(question, context)[:420], '...\n')

def mock_llm(prompt):
    return 'Customers may request a refund within 30 days of purchase [1].'
answer = mock_llm(rag_prompt(question, context))
print('answer:', answer)
print('invalid citations:', invalid_citations(answer, len(used)))
print('groundedness proxy:', groundedness(answer, [c.text for c in used]))
print('hallucinated answer groundedness:', groundedness('Refunds are paid in gold within 3 hours.', [c.text for c in used]))

## 5. Evaluate retrieval
Label which chunks answer each query, then compare strategies with Recall@k / MRR. Tiny sets are only illustrative.

In [ ]:
def find(sub):  # ids of chunks containing a phrase
    return {c.id for c in chunks if sub in c.text}
gold = {
 'refund within 30 days': find('refund within 30 days'),
 'ERR-4012 invoice address': find('ERR-4012'),
 'rotated every 90 days passwords': find('rotated every 90 days'),
 'meal expenses reimbursed receipts': find('Meal expenses'),
 'report phishing security desk': find('phishing'),
}
def run(name, fn):
    rec, rr = [], []
    for q, rel in gold.items():
        ids = [c.id for c in fn(q)]
        rec.append(recall_at_k(ids, rel, 3)); rr.append(reciprocal_rank(ids, rel))
    print(f'{name:7s} Recall@3={np.mean(rec):.2f}  MRR={np.mean(rr):.2f}')
run('dense', lambda q: [c for c, _ in store.search(q, 5)])
run('bm25',  lambda q: [c for c, _ in bm.search(q, 5)])
run('hybrid', lambda q: hybrid_search(store, bm, q, 5))